# Revisions: plotting

In [ ]:
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import os

import matplotlib.pyplot as plt
from matplotlib.pyplot import rc_context
import seaborn as sns
import warnings

from inmoose.pycombat import pycombat_norm, pycombat_seq

In [ ]:
# Set publication-quality parameters
plt.rcParams['pdf.fonttype'] = 42  # Makes text editable in PDF
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
plt.rcParams['font.size'] = 10
plt.rcParams['axes.linewidth'] = 1.

In [ ]:
# Cell type renamer

cell_type_renamer = {
  "Adipocyte"   : "Adipocyte",
  "Bcell"       : "Bcell",
  "CD4_CCR6"    : "CD4 Tcell CCR6+",
  "CD4_GZMB"    : "CD4 Tcell GZMB+",
  "CD4_Other"   : "CD4 Tcell other",
  "CD4_Tex"     : "CD4 Tex",
  "CD4_Tmem"    : "CD4 Tmem",
  "CD4_Tn"      : "CD4 Tnaive",
  "CD4_Treg"    : "CD4 Treg",
  "CD8_GZMB"    : "CD8 Tcell GZMB+",
  "CD8_naive"   : "CD8 Tnaive",
  "CD8_Other"   : "CD8 Tcell other",
  "CD8_Tex"     : "CD8 Tex",
  "CD8_Tmem"    : "CD8 Tmem",
  "DC"          : "DC",
  "EC"          : "EC",
  "Lin-"        : "Unclassified",
  "Mac_M1"      : "M2-like macrophage",
  "Mac_M2"      : "M1-like macrophage",
  "MKs"         : "MK",
  "Mono_CD14"   : "Monocytes CD14+",
  "Mono_CD16"   : "Monocytes CD16+",
  "Mye_HLADR"   : "Myeloid HLA-DR+",
  "Mye_other"   : "Myeloid other",
  "Myeloma"     : "Myeloma",
  "NK"          : "NK cells",
  "Myeloid":"Myeloid",
  "T":"Tcell"
}

## Import

In [ ]:
adata_raw = sc.read('/mnt/disks/data/imc/CART_cohort/batch_repheno/in/full_data-no_uns.h5ad') #`.uns['spatial']` removed
adata_raw

In [ ]:
# Labels

print(adata_raw.obs['sample_id'].shape[0])
adata_raw.obs = adata_raw.obs.reset_index().merge(
    pd.read_csv('/mnt/disks/data/imc/CART_cohort/batch_repheno/out/v2_pheno-labels_update3.csv',index_col=0)
).set_index('index')
df_=adata_raw.obs
print(df_.shape[0])
print(df_['lineage'].isna().sum())
print(df_['ph'].isna().sum())

In [ ]:
adata=adata_raw

In [ ]:
# Normalisation/batch correction

# Remove adipo, null expression confuses
adata_adipo = adata[adata.obs.lineage=='Adipocyte',]
adata = adata[adata.obs.lineage!='Adipocyte',]

# Total signal, log1p
adata_adipo.obs['log1p_totalsignal'] = np.log1p(adata_adipo.X.sum(axis=1))
adata.obs['log1p_totalsignal'] = np.log1p(adata.X.sum(axis=1))

# Additional 0-aware norm
#'log1p(X+1/log1p_totalsignal)'
adata.layers['Xpc_totsig_log1p_combat'] = pycombat_norm(
    np.log1p(pd.DataFrame(pd.DataFrame(adata.X).fillna(0).values.T+1).div(adata.obs['log1p_totalsignal'].tolist(), axis=1)),
    adata.obs.cohort).T

adata_adipo.layers['Xpc_totsig_log1p_combat'] = adata_adipo.X

adata = ad.concat([adata_adipo,adata], axis=0)

In [ ]:
adata

## Combined heatmap

In [ ]:
markers = ['aSMA','CD31','CD45','CD11b','CD68','CD163','CD14','CD16','HLA-DR','CD11c',
           'CD3','CD4','CD8','CD127','CD45RO','Granzyme_B','PD-1','CCR6','FoxP3','TIM-3','CD56','CD20','CD138']

adata_pl = adata[~adata.obs['lineage'].isin(['B-T','Artifact','Adipocyte']),markers]

adata_pl.obs['label'] = [cell_type_renamer[c] for c in adata_pl.obs['ph']]

adata_pl.obs.label.unique()

In [ ]:
label_order = ['EC', 'MK', 
               'M1-like macrophage','M2-like macrophage','Monocytes CD14+','Monocytes CD16+','Myeloid HLA-DR+','Myeloid other','DC',
               'CD4 Tnaive','CD4 Tmem','CD4 Tcell GZMB+','CD4 Tex','CD4 Tcell CCR6+','CD4 Treg','CD4 Tcell other',
               'CD8 Tnaive','CD8 Tmem','CD8 Tcell GZMB+','CD8 Tex','CD8 Tcell other',
               'NK cells', 'Bcell', 'Myeloma', 'Unclassified']

In [ ]:
markers2label = {
    'EC':'aSMA',
    'MK':'CD31',
    'Immune':'CD45',
    'Myeloid:':['CD11b','CD68','CD163','CD14','CD16','HLA-DR','CD11c'],
    'T cell':['CD3','CD4','CD8','CD127','CD45RO','Granzyme_B','PD-1','CCR6','FoxP3','TIM-3'],
    'NK cell':'CD56',
    'B cell':'CD20',
    'Myeloma':'CD138'
}

In [ ]:
fig = plt.figure(figsize=(8, 6), dpi=300)

sc.pl.matrixplot(adata_pl, markers2label, 'label', layer='Xpc_totsig_log1p_combat',
                 categories_order=label_order,
                 dendrogram=False, cmap="coolwarm",swap_axes=False,standard_scale='var',show=False)
plt.tight_layout()
plt.savefig('/mnt/disks/data/imc/CART_cohort/batch_repheno/pl/hm-all.pdf', format='pdf',dpi=300,bbox_inches='tight',transparent=False)
plt.savefig('/mnt/disks/data/imc/CART_cohort/batch_repheno/pl/hm-all.svg', format='svg',dpi=300,bbox_inches='tight',transparent=False)
plt.show()